# *TASK 2: Recommendation System with Streamlit GUI*

<p style='font-size:20px';><i>With the growing volume of content, users often struggle to find relevant items. The objective is to build a recommendation system that suggests similar movies based on user input using content-based filtering.</i></p>

In [1]:
# Import the necessary libaries
import pandas as pd
# Load the dataset
df = pd.read_csv("movies.csv")
# Display the dataset
df.head()

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


In [2]:
# Replace | with space
df['genres'] = df['genres'].str.replace('|', ' ')

# Remove nulls
df.dropna(inplace=True)

In [3]:
# Import TfidfVectorizer from scikit-learn's feature extraction module
from sklearn.feature_extraction.text import TfidfVectorizer
# Initialize the TF-IDF vectorizer with English stop words removed
tfidf = TfidfVectorizer(stop_words='english')
# Transform the 'genres' column into a TF-IDF matrix
tfidf_matrix = tfidf.fit_transform(df['genres'])

In [4]:
# Import the cosine_similarity function from scikit-learn's metrics.pairwise module
from sklearn.metrics.pairwise import cosine_similarity
# Calculate the cosine similarity between each pair of documents in the TF-IDF matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [5]:
# Create a Series where the index is the 'title' column and the values are the original dataframe indices
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

In [6]:
def recommend_movies(title, cosine_sim=cosine_sim):
    
    if title not in indices:
        return ["Movie not found"]
    
    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort by similarity
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Top 5 similar (excluding itself)
    sim_scores = sim_scores[1:6]

    movie_indices = [i[0] for i in sim_scores]
    similarity_scores = [i[1] for i in sim_scores]

    results = df['title'].iloc[movie_indices].tolist()

    return list(zip(results, similarity_scores))

In [7]:
print(recommend_movies("Toy Story (1995)"))

[('Antz (1998)', 1.0000000000000002), ('Toy Story 2 (1999)', 1.0000000000000002), ('Adventures of Rocky and Bullwinkle, The (2000)', 1.0000000000000002), ("Emperor's New Groove, The (2000)", 1.0000000000000002), ('Monsters, Inc. (2001)', 1.0000000000000002)]


In [8]:
import joblib

joblib.dump(cosine_sim, "cosine_sim.pkl")
joblib.dump(df, "movies.pkl")
joblib.dump(indices, "indices.pkl")

['indices.pkl']

In [ ]:
!streamlit run app1.py

# *Conclusion*

<p style='font-size:20px';><i>The recommendation system effectively demonstrated how similarity-based techniques can personalize user experience. The integration with Streamlit provided an interactive and practical interface for recommendations.</i></p>